In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using LorentzianSimplexSolver

In [ ]:
# Numerical precision and input data
ScalarT = Float64
# ScalarT = BigFloat

tol = configure_precision!(ScalarT; precision=256)
gamma_val = ScalarT(1.0)

simplices = [[2,3,4,6,7],[1,3,4,6,7],[1,2,4,6,7],[1,2,3,6,7],[1,2,3,4,7],[2,3,5,6,8],[1,3,5,6,8],[1,2,5,6,8],[1,2,3,6,8],[1,2,3,5,8],[2,4,5,6,9],[1,4,5,6,9],[1,2,5,6,9],[1,2,4,6,9],[1,2,4,5,9],[3,4,5,6,10],[1,4,5,6,10],[1,3,5,6,10],[1,3,4,6,10],[1,3,4,5,10],[3,4,5,6,11],[2,4,5,6,11],[2,3,5,6,11],[2,3,4,6,11],[2,3,4,5,11]]

ns = length(simplices)
all_vertices = sort(unique(Iterators.flatten(simplices)))
Nverts = length(all_vertices)

coords_lines = [
    "0, 0, 0, 0",
    "0, -2.7745276335252114, -0.9809436521275706, -1.6990442448471226",
    "0, 0, 0, -3.398088489694245",
    "-0.24028114141347542, -0.6936319083813028, -0.9809436521275706, -1.6990442448471226",
    "0, 0, -2.942830956382712, -1.6990442448471226",
    "-0.068, -0.27, -0.5, -1.3",
    "-0.06165622828269508, -0.7476319083813029, -0.49237746085102824, -1.6192353958776982",
    "-0.013600000000000001, -0.6089055267050423, -0.8847549217020566, -1.6192353958776982",
    "-0.05471634704156723, -0.6634799624836852, -1.2905133626305172, -1.3266576479993824",
    "-0.05996263164968173, -0.18743250119993968, -0.8604521717798855, -1.5747576871100133",
    "-0.0423034122998675, -0.5046526423930288, -1.0289336294868097, -2.25080216236223",
]

@assert length(coords_lines) == Nverts

vertex_coords = Dict{Int, Vector{ScalarT}}(
    v => LorentzianSimplexSolver.PrecisionUtils.parse_numeric_line(line, ScalarT)
    for (v, line) in zip(all_vertices, coords_lines)
)

(simplices=ns, vertices=Nverts, tolerance=tol)

In [ ]:
# Construct one GeometryCollection from the simplices and coordinates.
geom = construct_geometry(simplices, vertex_coords; verbose=false);
length(geom.simplex)

In [ ]:
# Build global connectivity, match shared faces, and apply gauge fixing.
# For a single simplex this function safely returns without changing the workflow.
prepare_global_geometry!(
    geom,
    simplices;
    sector=:ref,
    match_faces=true,
    gauge_fix=true,
    check=false,
    verbose=false,
);

isempty(geom.connectivity) ? "single simplex" : keys(geom.connectivity[1])

In [ ]:
# Optional checks for debugging.
# check_simplex_consistency(geom)
# LorentzianSimplexSolver.FaceMatchingChecks.check_all(geom)

In [ ]:
# Regge action and the geometric data entering the boundary phase.
regge = compute_regge_action(geom, simplices, vertex_coords);
regge.iregge

In [ ]:
# Spinfoam action at the critical point.
# The objects S and sd are kept as convenient debugging handles.
spinfoam = compute_spinfoam_action(geom, regge; gamma=gamma_val);
S = spinfoam.action_no_phase;
sd = spinfoam.solve_data;
spinfoam.action

In [ ]:
# Equations of motion at the critical point.
dS = check_eom(spinfoam; gamma=gamma_val);

In [ ]:
# Hessian computation can be expensive for large complexes.
# Switch this flag on when you need the quadratic fluctuation matrix.
compute_hessian_now = false

if compute_hessian_now
    hessian = compute_hessian(geom, spinfoam; gamma=gamma_val, eigenvalues=true)
    size(hessian.matrix), hessian.eigenvalues
end

In [ ]:
# Useful debugging handles after running the cells above:
# geom
# regge
# spinfoam
# dS
# hessian